# 04 — How the six datasets relate

This notebook treats the files like related tables in a database. It identifies the matching columns, demonstrates the two exact-key links, then measures how well AIS observations can be paired with the three environmental grids. The environmental links need nearest location and nearest time because the grids do not contain vessel IDs and their coordinates/timestamps do not exactly match each AIS report.

The first pass uses a reproducible sample of AIS positions to measure coverage. It does not create a large joined modelling table or choose a final matching rule. The match distance and time offset are retained so the trade-offs can be reviewed before any analysis uses these links.

**Relationship map**

- AIS_POS → AIS_SPEC: `Mmsi` plus the latest AIS_SPEC report at or before the position time. This is a time-aware many-to-one lookup; MMSI alone would match a vessel to every static report it ever sent.
- AIS_SPEC → AIS_ShipTypes: exact `ShipType` = `Code`. This is a normal lookup-table join.
- AIS_POS → ERA5, CMEMS_PHY, CMEMS_WAV: nearest usable grid cell and nearest available timestamp, subject to explicit time limits. This is a measured spatial-temporal match, not an exact SQL equality join.

The sample and thresholds below are for coverage diagnostics. The final environmental match rule should be chosen after reviewing the distance and time-offset distributions, grid masks, and study-area edges.

## 1. Load the cleaned tables and inspect their join fields

Cleaned copies are used so the meanings already assigned to timestamps and coded AIS values stay consistent with the earlier notebooks. Geometry is not needed for this first coverage check: latitude and longitude columns are used directly.

In [1]:
# Import pathlib for project-relative file paths.
from pathlib import Path
# Import NumPy for coordinate calculations and reproducible sampling.
import numpy as np
# Import pandas for reading, inspecting, and joining the tables.
import pandas as pd
# Import display so tables are easy to inspect in Jupyter.
from IPython.display import display
# Find the project folder from this notebook's location.
PROJECT = Path.cwd()
# Move up one level when Jupyter was started inside the notebooks folder.
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent
# Point to the reproducibly cleaned files created by the cleaning script.
PROCESSED = PROJECT / 'data' / 'processed'
# Load AIS positions and keep one row per confirmed unique MMSI/report time.
ais = pd.read_parquet(PROCESSED / 'ais_positions_clean.parquet', columns=['Mmsi', 'Date_utc', 'Latitude', 'Longitude', 'position_valid_flag', 'sog_knots_clean', 'sog_message27_unavailable_flag', 'cog_deg_clean', 'NavigationStatus'])
# Load AIS static/voyage reports, which can contain several reports per MMSI.
spec = pd.read_parquet(PROCESSED / 'ais_specifications_clean.parquet', columns=['Mmsi', 'Date_utc', 'ShipType', 'ship_type_label', 'draught_m_clean'])
# Load the compact ship-type code lookup.
ship_types = pd.read_csv(PROCESSED / 'ais_ship_types_clean.csv')
# Load each environmental grid for the coverage calculations below.
era5 = pd.read_parquet(PROCESSED / 'era5_clean.parquet', columns=['time_utc', 'latitude', 'longitude', 'sp', 'u10', 'v10'])
# Load the hourly near-surface ocean grid.
phy = pd.read_parquet(PROCESSED / 'cmems_phy_clean.parquet', columns=['time_utc', 'latitude', 'longitude', 'thetao', 'uo', 'vo'])
# Load the three-hour wave grid.
wav = pd.read_parquet(PROCESSED / 'cmems_wav_clean.parquet', columns=['time_utc', 'latitude', 'longitude', 'VHM0'])
# Show each table's row count and the columns used to connect it.
relationship_fields = pd.DataFrame([
    {'table': 'AIS_POS', 'rows': len(ais), 'link fields': 'Mmsi, Date_utc, Latitude, Longitude'},
    {'table': 'AIS_SPEC', 'rows': len(spec), 'link fields': 'Mmsi, Date_utc, ShipType'},
    {'table': 'AIS_ShipTypes', 'rows': len(ship_types), 'link fields': 'Code'},
    {'table': 'ERA5', 'rows': len(era5), 'link fields': 'time_utc, latitude, longitude'},
    {'table': 'CMEMS_PHY', 'rows': len(phy), 'link fields': 'time_utc, latitude, longitude'},
    {'table': 'CMEMS_WAV', 'rows': len(wav), 'link fields': 'time_utc, latitude, longitude'},
])
# Display the table sizes and the fields that take part in matching.
display(relationship_fields)
# Confirm that cleaned AIS report times are timezone-aware UTC timestamps.
print('AIS_POS Date_utc dtype:', ais['Date_utc'].dtype)
# Confirm that environmental grid times use the same UTC convention.
print('ERA5 time_utc dtype:', era5['time_utc'].dtype)

,table,rows,link fields
0,AIS_POS,18973676,"Mmsi, Date_utc, Latitude, Longitude"
1,AIS_SPEC,13547406,"Mmsi, Date_utc, ShipType"
2,AIS_ShipTypes,256,Code
3,ERA5,537510,"time_utc, latitude, longitude"
4,CMEMS_PHY,5007803,"time_utc, latitude, longitude"
5,CMEMS_WAV,1670779,"time_utc, latitude, longitude"


AIS_POS Date_utc dtype: datetime64[ns, UTC]
ERA5 time_utc dtype: datetime64[ns, UTC]


## 2. Exact lookup join: decode the ship-type code

This is the familiar SQL pattern: a code in one table points to one row in a lookup table. Before joining, check that the lookup key is unique; otherwise one AIS report could unexpectedly turn into multiple rows. Code 0 is kept as a label for now and remains “not available,” not a vessel category for comparisons.

In [2]:
# Check that each ship-type code identifies no more than one lookup row.
print('Duplicate lookup codes:', int(ship_types['Code'].duplicated().sum()))
# Read only the fields needed to demonstrate the vessel-type relationship.
spec_type_codes = spec[['Mmsi', 'Date_utc', 'ShipType']].copy()
# Match each AIS type code to its human-readable lookup description.
spec_with_type = spec_type_codes.merge(
    ship_types[['Code', 'Type']],
    left_on='ShipType',
    right_on='Code',
    how='left',
    validate='many_to_one',
)
# Count source rows whose type code did not find a lookup description.
unmatched_types = int(spec_with_type['Type'].isna().sum())
# Show the lookup coverage and a few joined rows for inspection.
print(f'AIS_SPEC rows: {len(spec_type_codes):,}; rows without a label: {unmatched_types:,}')
# Display example code-to-label results.
display(spec_with_type[['Mmsi', 'ShipType', 'Type']].head(10))

Saved row-level preview omitted from the public copy to avoid republishing individual source records. Run this cell locally with the source files to inspect the rows.


## 3. Time-aware AIS join: attach the latest vessel report known at that time

AIS_SPEC is a stream of changing reports, rather than one permanent row per vessel. A position should therefore use the most recent specification report sent by the same MMSI at or before its timestamp. The age of that report is measured and reported: an old destination or draught should not silently be treated as current. This is an as-of join, equivalent to an SQL lookup with matching MMSI and a maximum report time no later than the position time.

To keep this demonstration quick, one position per MMSI is sampled with a fixed random seed. The result measures the age and availability of linked specifications; it does not yet pick an acceptable age limit.

In [3]:
# Use a fixed seed so the same MMSI sample can be reproduced.
RANDOM_SEED = 42
# Keep a manageable sample of unique vessels for the as-of join demonstration.
ais_vessel_sample = ais.groupby('Mmsi', group_keys=False).sample(n=1, random_state=RANDOM_SEED)
# Keep only useful specification fields and rename them to show their origin.
spec_link = spec[[
    'Mmsi', 'Date_utc', 'ShipType', 'ship_type_label', 'draught_m_clean',
]].rename(columns={
    'Date_utc': 'spec_time_utc',
    'ShipType': 'spec_ship_type_code',
    'ship_type_label': 'spec_ship_type_label',
    'draught_m_clean': 'spec_draught_m',
})
# Sort both sides by time because pandas requires ordered keys for an as-of join.
ais_for_asof = ais_vessel_sample.sort_values('Date_utc').copy()
# Sort vessel reports by time and MMSI to make the most recent prior report available.
spec_for_asof = spec_link.sort_values('spec_time_utc').copy()
# Match within MMSI and take the latest report not later than the AIS position.
ais_spec_sample = pd.merge_asof(
    ais_for_asof,
    spec_for_asof,
    left_on='Date_utc',
    right_on='spec_time_utc',
    by='Mmsi',
    direction='backward',
)
# Calculate how long ago the linked vessel-information report was received.
ais_spec_sample['spec_age_days'] = (
    ais_spec_sample['Date_utc'] - ais_spec_sample['spec_time_utc']
).dt.total_seconds() / 86400
# Summarize linkage and report age before deciding whether any age cutoff is justified.
print('Sampled AIS positions:', f'{len(ais_spec_sample):,}')
print('Positions with an earlier AIS_SPEC report:', f"{ais_spec_sample['spec_time_utc'].notna().sum():,}")
# Display report-age quantiles so stale links can be recognized.
display(ais_spec_sample['spec_age_days'].describe(percentiles=[.5, .9, .95, .99]).to_frame('age_days'))
# Show how often linked draught and ship type are actually populated.
display(ais_spec_sample[['spec_ship_type_label', 'spec_draught_m']].notna().mean().rename('share populated').to_frame())

Sampled AIS positions: 25,130
Positions with an earlier AIS_SPEC report: 23,484


,age_days
count,23484.000000
mean,0.350658
std,3.062729
min,0.000000
50%,0.008322
90%,0.048307
95%,0.188119
99%,10.048642
max,79.578495


,share populated
spec_ship_type_label,0.934501
spec_draught_m,0.311023


## 4. Environmental matching: nearest usable grid location and nearest time

The environmental tables have no MMSI. Each AIS point needs to be paired with the closest usable grid location and the closest grid timestamp. Coordinates are converted to points on a sphere before calculating distance, so longitude degrees are not incorrectly treated as the same physical distance everywhere. For each dataset, cells with no usable value for its representative field are excluded from the location search. The resulting distance and time offset stay attached to every match.

The diagnostic limits below are half of the expected time interval: 30 minutes for hourly ERA5/CMEMS_PHY and 90 minutes for three-hour CMEMS_WAV. They prevent a position from being called matched to a distant time step. Spatial distance is reported first and has no acceptance cutoff in this diagnostic; we will choose that after seeing the results and map extent.

This is like a two-part SQL join: find a grid-cell key from location, find a time key from the nearest timestamp, then join on those two keys. The final merge uses integer keys created from the grid coordinates and time axis, which avoids floating-point equality problems.

In [4]:
# Import a spatial tree so each AIS point can quickly find its nearest grid point.
from scipy.spatial import cKDTree
# Define the Earth's mean radius in kilometres for converting angular distance.
EARTH_RADIUS_KM = 6371.0088
# Set the allowed time difference to half an hourly grid interval.
HOURLY_TIME_LIMIT = pd.Timedelta(minutes=30)
# Set the allowed time difference to half a three-hour wave-grid interval.
WAVE_TIME_LIMIT = pd.Timedelta(minutes=90)
# Keep a deterministic sample of valid position events across the full study period.
ais_match_sample = ais.loc[ais['position_valid_flag']].sample(
    n=min(50000, int(ais['position_valid_flag'].sum())),
    random_state=RANDOM_SEED,
).copy()
# Convert latitude/longitude points to 3D unit-sphere coordinates for distance search.
def to_unit_sphere(latitude, longitude):
    # Convert degrees to radians because trigonometric functions use radians.
    lat_rad = np.radians(np.asarray(latitude, dtype='float64'))
    # Convert longitude to radians using the same angular convention.
    lon_rad = np.radians(np.asarray(longitude, dtype='float64'))
    # Return x, y, z points on a unit sphere.
    return np.column_stack((np.cos(lat_rad) * np.cos(lon_rad), np.cos(lat_rad) * np.sin(lon_rad), np.sin(lat_rad)))
# Match a sample to one environmental grid and return coverage diagnostics.
def match_grid_coverage(position_sample, grid, dataset_name, representative_field, time_limit, measurement_fields):
    # Keep grid coordinates where the representative measurement exists at least once.
    usable_rows = grid.loc[grid[representative_field].notna(), ['time_utc', 'latitude', 'longitude']].copy()
    # Build the list of spatial cells that have at least one usable observation.
    cells = usable_rows[['latitude', 'longitude']].drop_duplicates().reset_index(drop=True)
    # Build a fast nearest-cell search structure from the usable cell coordinates.
    tree = cKDTree(to_unit_sphere(cells['latitude'], cells['longitude']))
    # Find the nearest usable cell for every sampled vessel position.
    chord_distance, cell_id = tree.query(to_unit_sphere(position_sample['Latitude'], position_sample['Longitude']), k=1)
    # Convert unit-sphere chord distance to great-circle distance in kilometres.
    distance_km = 2 * EARTH_RADIUS_KM * np.arcsin(np.clip(chord_distance / 2, 0, 1))
    # Make a table of position rows and the matched spatial-cell key.
    matched = position_sample[['Mmsi', 'Date_utc', 'Latitude', 'Longitude', 'sog_knots_clean', 'NavigationStatus']].reset_index(drop=True).copy()
    # Attach the chosen cell number and its distance from the AIS report.
    matched['_cell_id'] = cell_id
    # Keep spatial distance visible for every later coverage decision.
    matched['grid_distance_km'] = distance_km
    # Add the selected grid coordinates so the join key can be inspected.
    matched = matched.join(cells.add_prefix('grid_'), on='_cell_id')
    # Create a sorted list of timestamps available in this environmental file.
    times = pd.DatetimeIndex(grid['time_utc'].drop_duplicates().sort_values())
    # Find the insertion point of every AIS time in the environmental time axis.
    insertion = times.get_indexer(matched['Date_utc'], method='nearest', tolerance=None)
    # Get the nearest environmental time; indexer returns -1 only for an empty axis.
    matched['_time_id'] = insertion
    # Attach the selected time as a column for measuring the exact offset.
    matched['grid_time_utc'] = times.take(insertion)
    # Measure signed time offset in minutes; positive means the grid time is later.
    matched['time_offset_minutes'] = (matched['grid_time_utc'] - matched['Date_utc']).dt.total_seconds() / 60
    # Make a temporary integer cell ID on the full environmental table.
    cell_lookup = cells.reset_index().rename(columns={'index': '_cell_id'})
    # Add each environmental row's cell ID by matching its coordinates.
    grid_keyed = grid.merge(cell_lookup, on=['latitude', 'longitude'], how='inner', validate='many_to_one')
    # Map each timestamp to its position in the sorted time axis.
    time_lookup = pd.DataFrame({'time_utc': times, '_time_id': np.arange(len(times))})
    # Attach the integer time key to each environmental record.
    grid_keyed = grid_keyed.merge(time_lookup, on='time_utc', how='inner', validate='many_to_one')
    # Retain representative value and an availability marker for the matched field.
    grid_values = grid_keyed[['_cell_id', '_time_id'] + measurement_fields].copy()
    # Several AIS reports can share one grid cell and timestamp, so this is many-to-one.
    matched = matched.merge(grid_values, on=['_cell_id', '_time_id'], how='left', validate='many_to_one')
    # Mark time matches that fall within the documented half-interval limit.
    matched['within_time_limit'] = (matched['time_offset_minutes'].abs() <= time_limit.total_seconds() / 60)
    # Mark whether the representative environmental measurement is available in that cell/time.
    matched['measurement_available'] = matched[representative_field].notna()
    # Summarize raw time coverage and actual measurement availability separately.
    summary = {
        'dataset': dataset_name,
        'sample_positions': len(matched),
        'within_time_limit_pct': 100 * matched['within_time_limit'].mean(),
        'measurement_available_pct': 100 * (matched['within_time_limit'] & matched['measurement_available']).mean(),
        'median_grid_distance_km': matched['grid_distance_km'].median(),
        'p95_grid_distance_km': matched['grid_distance_km'].quantile(.95),
        'median_abs_time_offset_min': matched['time_offset_minutes'].abs().median(),
        'p95_abs_time_offset_min': matched['time_offset_minutes'].abs().quantile(.95),
    }
    # Return row-level diagnostics and a compact summary for comparison.
    return matched, summary

## 5. Measure match coverage for each environmental file

Each grid has a representative field used to detect cells with usable data: surface pressure for ERA5, potential temperature for CMEMS_PHY, and significant wave height for CMEMS_WAV. The selected nearest grid cell and time are then checked for the actual value. This exposes a coastal or land-mask gap as missing coverage rather than hiding it. The chosen field is only a coverage indicator; later analyses may require several variables to be present together.

In [5]:
# Match AIS positions to the coarser hourly weather grid, using pressure as its coverage field.
era5_matches, era5_summary = match_grid_coverage(ais_match_sample, era5, 'ERA5', 'sp', HOURLY_TIME_LIMIT, ['sp', 'u10', 'v10'])
# Match positions to hourly near-surface ocean data, using potential temperature as its coverage field.
phy_matches, phy_summary = match_grid_coverage(ais_match_sample, phy, 'CMEMS_PHY', 'thetao', HOURLY_TIME_LIMIT, ['thetao', 'uo', 'vo'])
# Match positions to the three-hour wave grid, using significant wave height as its coverage field.
wav_matches, wav_summary = match_grid_coverage(ais_match_sample, wav, 'CMEMS_WAV', 'VHM0', WAVE_TIME_LIMIT, ['VHM0'])
# Put all three coverage summaries side by side for an easy comparison.
coverage = pd.DataFrame([era5_summary, phy_summary, wav_summary]).set_index('dataset')
# Display percentage matched, spatial distance, and time offset for each grid.
display(coverage.round(2))
# Inspect example weather matches with their distance and time offset retained.
display(era5_matches[['Mmsi', 'Date_utc', 'grid_time_utc', 'time_offset_minutes', 'grid_distance_km', 'sp', 'within_time_limit']].head(10))

Saved row-level preview omitted from the public copy to avoid republishing individual source records. Run this cell locally with the source files to inspect the rows.


## 6. First AIS-to-environment association check

### Why this analysis matters

The intended user is a fleet operations or voyage performance analyst reviewing vessel movements. The practical use is to provide a context-aware reference for observed speed, so recurring slow-speed areas or unusual segments can be prioritized for human review after a voyage. A flagged segment is a prompt to check weather, traffic, manoeuvring, schedule, or data quality; it does not identify the cause by itself.

This is a screening and review use case. AIS speed and environmental conditions alone cannot recommend a safe or optimal speed, prove fuel savings, or measure emissions. Those require vessel-specific operating limits, fuel/engine data, voyage plans, and schedule constraints.

### Research question and hypothesis

**Research question:** Do wind, current, and wave measurements improve estimates of observed speed over ground beyond a baseline that accounts for vessel category, route/location, time, and local AIS traffic?

**H0:** Adding environmental measurements does not improve speed estimates on held-out vessels or periods compared with the context-only baseline.

**H1:** Adding environmental measurements improves those estimates, and the improvement is reasonably stable across sensible spatial and time matching choices.

The first correlations below are descriptive only. They do not test H0 because vessel, route, and traffic context have not yet been controlled. They also use repeated reports from the same vessels, so they are not independent evidence or a causal test.

### Candidate model

Regression is the natural first model because speed is measured on a continuous scale. Compare two versions: a baseline with vessel category, route/location, time, and AIS-derived local traffic; then the same inputs plus wind, current, and wave context. Use cleaned speed over ground in knots as the target after defining which reports represent a vessel underway. Evaluate on vessels kept out of model fitting, and use a later-time check where feasible. Report whether environmental inputs reduce prediction error and by how much. Standardize numeric inputs for a regularized linear model inside a training-only pipeline; a tree-based regressor can be compared without that scaling. A small improvement would mean these environmental grids add limited operational context at their current resolution.

In [6]:
# Start with AIS positions and the three environmental match results in their shared sample order.
joint_sample = ais_match_sample[['Mmsi', 'Date_utc', 'Latitude', 'Longitude', 'sog_knots_clean', 'sog_message27_unavailable_flag', 'cog_deg_clean', 'NavigationStatus']].reset_index(drop=True).copy()
# Attach the weather measurements, preserving their match distance and time offset.
for field in ['sp', 'u10', 'v10']:
    joint_sample[f'era5_{field}'] = era5_matches[field].to_numpy()
joint_sample['era5_distance_km'] = era5_matches['grid_distance_km'].to_numpy()
joint_sample['era5_within_time'] = era5_matches['within_time_limit'].to_numpy()
# Attach the ocean measurements and their corresponding match diagnostics.
for field in ['thetao', 'uo', 'vo']:
    joint_sample[f'phy_{field}'] = phy_matches[field].to_numpy()
joint_sample['phy_distance_km'] = phy_matches['grid_distance_km'].to_numpy()
joint_sample['phy_within_time'] = phy_matches['within_time_limit'].to_numpy()
# Attach significant wave height and the wave-grid match diagnostics.
joint_sample['wave_VHM0_m'] = wav_matches['VHM0'].to_numpy()
joint_sample['wave_distance_km'] = wav_matches['grid_distance_km'].to_numpy()
joint_sample['wave_within_time'] = wav_matches['within_time_limit'].to_numpy()
# Calculate wind speed from eastward and northward wind components in metres per second.
joint_sample['wind_speed_ms'] = np.hypot(joint_sample['era5_u10'], joint_sample['era5_v10'])
# Calculate current speed from eastward and northward surface-current components in metres per second.
joint_sample['current_speed_ms'] = np.hypot(joint_sample['phy_uo'], joint_sample['phy_vo'])
# Identify rows within each product's time coverage before checking spatial sensitivity.
joint_sample['all_time_matches'] = (joint_sample['era5_within_time'] & joint_sample['phy_within_time'] & joint_sample['wave_within_time'])
# Count stationary/slow reports at several candidate thresholds rather than fixing one silently.
speed_threshold_counts = pd.DataFrame({
    'threshold_knots': [0.0, 0.5, 1.0, 2.0],
    'sample_reports_above_threshold': [int(joint_sample['sog_knots_clean'].gt(x).sum()) for x in [0.0, 0.5, 1.0, 2.0]],
})
# Show how the moving-report definition changes the available sample size.
display(speed_threshold_counts)
# Define candidate spatial limits for a coverage-versus-distance sensitivity check.
distance_limits_km = [10, 15, 20, 25, 30]
# Keep only rows with valid speed, required environmental measurements, and time matches.
required_fields = ['sog_knots_clean', 'wind_speed_ms', 'current_speed_ms', 'wave_VHM0_m']
# Build one summary row per distance limit without discarding the source sample.
sensitivity_rows = []
for max_distance_km in distance_limits_km:
    # Require each dataset's nearest cell to be within this candidate distance.
    distance_ok = (joint_sample['era5_distance_km'].le(max_distance_km) & joint_sample['phy_distance_km'].le(max_distance_km) & joint_sample['wave_distance_km'].le(max_distance_km))
    # Require the representative measurements and valid time matches from all products.
    complete = joint_sample['all_time_matches'] & distance_ok & joint_sample[required_fields].notna().all(axis=1)
    # Use 0.5 knots as a provisional moving threshold for this first look.
    moving = complete & joint_sample['sog_knots_clean'].gt(0.5)
    # Calculate rank correlations to compare monotonic association across cutoffs.
    rho = joint_sample.loc[moving, ['sog_knots_clean', 'wind_speed_ms', 'current_speed_ms', 'wave_VHM0_m']].corr(method='spearman')['sog_knots_clean']
    # Record coverage and association estimates, including the actual row count.
    sensitivity_rows.append({
        'max_distance_km': max_distance_km,
        'all_environment_matches_pct': 100 * complete.mean(),
        'moving_reports_n': int(moving.sum()),
        'rho_speed_wind': rho['wind_speed_ms'],
        'rho_speed_current': rho['current_speed_ms'],
        'rho_speed_wave_height': rho['wave_VHM0_m'],
    })
# Compare whether the early associations change as the distance rule tightens.
sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity.round(3))
# Show the speed distribution for the rows used at a provisional 20 km limit.
candidate_mask = (joint_sample['all_time_matches'] & joint_sample['era5_distance_km'].le(20) & joint_sample['phy_distance_km'].le(20) & joint_sample['wave_distance_km'].le(20) & joint_sample[required_fields].notna().all(axis=1) & joint_sample['sog_knots_clean'].gt(0.5))
display(joint_sample.loc[candidate_mask, 'sog_knots_clean'].describe(percentiles=[.5, .9, .95, .99]).to_frame('speed_over_ground_knots'))
# Treat these pooled correlations as exploratory because vessels contribute repeated reports.
print('Interpretation: compare direction and size across distance limits; do not treat these pooled values as causal or as final model evidence.')
# Save the reproducible, sampled AIS-to-environment table for the model notebook.
joint_sample_path = PROCESSED / 'joined_environment_sample.parquet'
# Keep this derived sample under the processed-data folder, which is excluded from Git.
joint_sample.to_parquet(joint_sample_path, index=False)
# Print the saved location and row count for the next notebook.
print(f'Saved {len(joint_sample):,} matched AIS positions to {joint_sample_path}')


,threshold_knots,sample_reports_above_threshold
0,0.0,23884
1,0.5,18924
2,1.0,18240
3,2.0,17276


,max_distance_km,all_environment_matches_pct,moving_reports_n,rho_speed_wind,rho_speed_current,rho_speed_wave_height
0,10,36.770,9847,0.136,0.120,0.169
1,15,75.586,16916,0.130,0.121,0.177
2,20,89.000,18241,0.134,0.124,0.187
3,25,95.290,18561,0.135,0.129,0.191
4,30,98.132,18664,0.135,0.130,0.192


,speed_over_ground_knots
count,18241.000000
mean,8.068132
std,4.751723
min,0.600000
50%,7.400000
90%,14.200000
95%,16.299999
99%,21.000000
max,44.799999


Interpretation: compare direction and size across distance limits; do not treat these pooled values as causal or as final model evidence.
Saved 50,000 matched AIS positions to /Users/midhunpraj/Documents/Maritime Traffic & Environmental Performance Analytics/data/processed/joined_environment_sample.parquet


## 7. Check the relationships before treating them as analysis data

A high percentage alone does not prove the match is suitable. Review long-distance links, observations near the grid boundary, wave-period offsets, and whether the variables required by a later question are jointly available. The final analysis can use the same position as the anchor and attach environmental variables with the recorded match distance and time difference. Keep unmatched rows and reasons in the coverage report so exclusions are visible.

At this stage, there are three distinct relationship types: an exact code lookup, a vessel-and-time lookup, and a spatial-temporal grid match. In database terms, they can all be represented with joins, but only the first is a simple equality join on a shared key. The grid matches need nearest-neighbour rules before the database join keys exist.

In [7]:
# Compare match distance quantiles to reveal whether coverage depends on very distant cells.
distance_check = pd.DataFrame({
    'ERA5_km': era5_matches['grid_distance_km'].quantile([0, .5, .9, .95, .99, 1]),
    'CMEMS_PHY_km': phy_matches['grid_distance_km'].quantile([0, .5, .9, .95, .99, 1]),
    'CMEMS_WAV_km': wav_matches['grid_distance_km'].quantile([0, .5, .9, .95, .99, 1]),
})
# Display distance quantiles using the same percentiles for all three grids.
display(distance_check.round(2))
# Compare signed time offsets to check whether nearest-time matching is balanced before and after AIS reports.
time_check = pd.DataFrame({
    'ERA5_minutes': era5_matches['time_offset_minutes'].describe(percentiles=[.05, .5, .95]),
    'CMEMS_PHY_minutes': phy_matches['time_offset_minutes'].describe(percentiles=[.05, .5, .95]),
    'CMEMS_WAV_minutes': wav_matches['time_offset_minutes'].describe(percentiles=[.05, .5, .95]),
})
# Show whether the selected environmental time tends to be earlier or later than AIS.
display(time_check.round(2))
# Count AIS sample positions that fall beyond each environmental file's time limits.
time_coverage = pd.DataFrame({
    'ERA5': era5_matches['within_time_limit'].value_counts(dropna=False),
    'CMEMS_PHY': phy_matches['within_time_limit'].value_counts(dropna=False),
    'CMEMS_WAV': wav_matches['within_time_limit'].value_counts(dropna=False),
}).fillna(0).astype(int)
# Display true/false counts so the numerator and denominator behind coverage are clear.
display(time_coverage)

,ERA5_km,CMEMS_PHY_km,CMEMS_WAV_km
0.00,0.12,0.00,0.00
0.50,9.42,3.97,4.69
0.90,15.48,14.60,13.81
0.95,17.64,20.91,16.08
0.99,24.41,26.81,17.79
1.00,41.38,33.02,28.10


,ERA5_minutes,CMEMS_PHY_minutes,CMEMS_WAV_minutes
count,50000.00,50000.00,50000.00
mean,-7.71,-0.10,0.11
std,85.67,17.35,51.99
min,-1436.67,-29.98,-89.98
5%,-27.63,-27.07,-81.07
50%,-0.47,-0.18,0.10
95%,26.97,27.02,80.98
max,30.00,30.00,90.00


,ERA5,CMEMS_PHY,CMEMS_WAV
False,524,0,0
True,49476,50000,50000


## What this establishes

The six files can be related through keys, but not all through the same kind of key. AIS positions anchor the project; vessel details attach through MMSI and report time; ship-type descriptions attach by exact code; environmental context attaches through nearest grid location and time. This creates a clear path to a combined analysis table while preserving the match quality for each row.

The pooled correlations in this notebook are an initial sample check, not the final answer to the research question. Before modelling, derive and inspect local AIS traffic, review high-speed observations, attach time-valid vessel categories, and settle the underway and spatial-match rules. Then compare the context-only baseline with the model that adds environmental conditions, keeping vessel or time groups out of model fitting as planned.